# PatchTST — Walmart Store Sales Forecasting

Model experiment notebook for the PatchTST branch of the multi-model
comparison (LightGBM, XGBoost, DLinear, TFT, PatchTST). PatchTST ("A Time
Series is Worth 64 Words: Long-Term Forecasting with Transformers", Nie et
al. 2023) segments each series' lookback window into a handful of patches
(like a Vision Transformer segments an image into patches), treats each
patch as one token, and runs a plain Transformer encoder over that short
patch sequence rather than over raw timesteps — a small number of tokens
(4 here) instead of 52, which is what makes it cheap relative to a
naive per-timestep transformer.

**Hand-rolled** (`utils/patchtst.py`), like DLinear and unlike TFT: the
architecture is simple enough — patching, a linear patch embedding, a plain
`nn.TransformerEncoder`, a linear output head — to validate correctly by
hand, with no LSTM, no gating/variable-selection networks, and no
static-covariate machinery to get wrong. Reuses `utils.dlinear`'s generic
(not DLinear-specific) data-prep helpers (`build_full_calendar_panel`,
`series_arrays_from_panel`, `build_aux_features`) rather than duplicating
them.

**Cells written but not executed** — meant to run on Colab. Every piece of
logic (patching math, RevIN normalization including negative-value and
constant-window edge cases, the training loop, the recursive multi-block
rollout, the raw-input pipeline including partial-request row alignment)
was independently validated against `utils/patchtst.py` with a throwaway
Python 3.11 CPU environment and synthetic data.

**Where this notebook sits relative to the other three:**

- **RevIN (instance normalization), not per-series stats.** DLinear and TFT
  both normalize each series using a single, precomputed mean/std for that
  whole series. PatchTST instead normalizes *each window* by *its own*
  lookback mean/std, denormalizing the output the same way — a real,
  validated part of the original architecture (it's how PatchTST handles
  distribution shift over time), kept faithful here rather than swapped
  for this project's usual per-series convention. One direct consequence:
  no `compute_series_stats`-equivalent step is needed at all — normalization
  lives inside the model's `forward()`, not in the data pipeline.
- **Future covariates added, unlike the original paper.** PatchTST as
  published is purely univariate with no covariate-injection mechanism —
  but this project's DLinear notebook found a real, visible demand spike
  (Easter) that a model with zero calendar signal completely misses, and
  fixed a real chunk of that gap by concatenating a future-known covariate
  channel. Same fix applied here: `IsHoliday` + week-of-year sin/cos are
  concatenated onto the flattened patch representation before the output
  head, using the exact same `utils.dlinear.build_aux_features` layout
  DLinear uses.
- **Direct 52-week horizon** for the holdout-eval and final production
  models, same reasoning as TFT: a linear output head maps to any horizon
  length in one shot, so there's no need for DLinear's recursive
  block-chaining (and the compounding error that comes with it). CV/tuning
  still uses a 13-week horizon, for the same 91-week `local_train_raw`
  budget reason every other notebook hit.
- **Same 3-split CV** (`[65, 72, 78]`-week boundaries) as DLinear and TFT —
  the 91-week budget constrains every architecture identically here,
  there's no reason to redesign it per notebook.

## Table of Contents
1. [Setup](#1)
2. [Local train/test split](#2)
3. [Sequence construction & validation splits](#3)
4. [Model architecture](#4)
5. [Hyperparameter tuning](#5)
6. [MLflow logging](#6)
7. [Plots](#7)
8. [Full pipeline](#8)

<a id='1'></a>
## 1. Setup

In [ ]:
import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore', category=DeprecationWarning)

from utils.patchtst import (
    PatchTST, PatchTSTWindowDataset, weighted_mae_loss, recursive_forecast, PatchTSTForecastPipeline,
    build_full_calendar_panel, series_arrays_from_panel,
)
from utils.feature_engineering import HOLIDAY_DATES
from utils.metrics import wmae

pd.set_option('display.max_columns', 50)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'

# features.csv/stores.csv intentionally not loaded, same choice DLinear made:
# PatchTST here is still a history(+holiday)-only model, not a tabular one.
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])

train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'train : {train.shape}, {train.Date.min().date()} -> {train.Date.max().date()}, '
      f'{train.Date.nunique()} weeks, {train[["Store","Dept"]].drop_duplicates().shape[0]} series')
print(f'test  : {test.shape}, {test.Date.min().date()} -> {test.Date.max().date()}, {test.Date.nunique()} weeks')

<a id='2'></a>
## 2. Local Train/Test Split

Identical split to every other notebook in this project.

In [ ]:
unique_dates = np.sort(train['Date'].unique())
cutoff_date = unique_dates[-52]

local_train_raw = train[train['Date'] < cutoff_date].copy()
local_test_raw = train[train['Date'] >= cutoff_date].copy()

print(f'cutoff date (first date held out): {pd.Timestamp(cutoff_date).date()}')
print(f'local_train_raw: {local_train_raw.shape}, {local_train_raw.Date.min().date()} -> {local_train_raw.Date.max().date()}  ({local_train_raw.Date.nunique()} weeks)')
print(f'local_test_raw : {local_test_raw.shape}, {local_test_raw.Date.min().date()} -> {local_test_raw.Date.max().date()}  ({local_test_raw.Date.nunique()} weeks)')

In [ ]:
def holidays_in_range(dates_series):
    dates_set = set(pd.to_datetime(dates_series))
    present = {}
    for name, dates in HOLIDAY_DATES.items():
        matched = [d for d in pd.to_datetime(dates) if d in dates_set]
        if matched:
            present[name] = [d.date() for d in matched]
    return present


print('local_train_raw holidays:', holidays_in_range(local_train_raw['Date']))
print('local_test_raw  holidays:', holidays_in_range(local_test_raw['Date']))
print()
print('Kaggle test.csv holidays (reference, no target):', holidays_in_range(test['Date']))

<a id='3'></a>
## 3. Sequence Construction & Validation Splits

`LOOKBACK = 52` weeks, `HORIZON_CV = 13` weeks for tuning (the same 91-week
`local_train_raw` budget constraint as DLinear/TFT — `CV_SPLIT_TRAIN_WEEKS
= [65, 72, 78]` is the identical spread, for the identical reason). No
per-series normalization step here (unlike DLinear's `compute_series_stats`)
— RevIN normalizes each window using its own lookback statistics, computed
inside `PatchTST.forward` itself, so `PatchTSTWindowDataset` deals in raw
(un-normalized) values throughout.

Every (Store, Dept) series present in a split's training range gets
reindexed onto the full weekly calendar and contributes sliding windows —
no series is excluded, same convention as every other notebook here.

In [ ]:
LOOKBACK = 52
HORIZON_CV = 13
HORIZON_FINAL = 52
CV_SPLIT_TRAIN_WEEKS = [65, 72, 78]

local_train_dates = np.sort(local_train_raw['Date'].unique())


def build_cv_split(train_end):
    tr_dates = local_train_dates[:train_end]
    va_dates = local_train_dates[train_end:train_end + HORIZON_CV]
    assert len(va_dates) == HORIZON_CV, 'val window must be exactly one HORIZON_CV block for a direct (non-recursive) validation score'

    tr_df = local_train_raw[local_train_raw['Date'].isin(tr_dates)]
    va_df = local_train_raw[local_train_raw['Date'].isin(va_dates)]

    panel = build_full_calendar_panel(tr_df)
    arrays = series_arrays_from_panel(panel)
    train_ds = PatchTSTWindowDataset(arrays, LOOKBACK, HORIZON_CV)

    val_panel = build_full_calendar_panel(pd.concat([tr_df, va_df], ignore_index=True))
    val_arrays = series_arrays_from_panel(val_panel)
    n_val_only = sum(1 for k in val_arrays if k not in arrays)

    return {
        'train_end': train_end, 'va_dates': va_dates,
        'train_ds': train_ds, 'val_arrays': val_arrays, 'n_val_only': n_val_only,
    }


cv_splits = [build_cv_split(te) for te in CV_SPLIT_TRAIN_WEEKS]

for s in cv_splits:
    print(f"split train_end={s['train_end']}: val {pd.Timestamp(s['va_dates'][0]).date()} -> {pd.Timestamp(s['va_dates'][-1]).date()} "
          f"holidays={list(holidays_in_range(s['va_dates']).keys())}")
    print(f"  {len(s['train_ds'])} training windows, {s['n_val_only']} val-only series with zero rows in the "
          f"training window (not skipped, unlike DLinear/TFT — RevIN normalizes their all-zero history slice fine)")

Baseline PatchTST (`patch_len=13, stride=13` -> 4 non-overlapping
patches, `d_model=16`, `n_heads=4`, `n_layers=2`, a fixed starter
`learning_rate=1e-3`) across all 3 splits, to confirm the training harness
works end-to-end before tuning (Step 5) builds on top of it — same purpose
as every other notebook's baseline step.

In [ ]:
def evaluate_val_wmae(model, train_weeks, val_arrays, lookback, horizon, device):
    """WMAE on ONE direct HORIZON-length prediction per series (no recursion —
    val_arrays' window from train_weeks to train_weeks+horizon is exactly one
    block by construction, see the assert above). Unlike DLinear/TFT, no
    per-series stats lookup here to crash on for series with no real
    training-period history (the KeyError DLinear originally hit) — RevIN
    computes its own mean/std from whatever hist_raw slice it's given, so an
    all-zero-history series just normalizes to ~0 and predicts off the aux
    covariates alone, no special-casing needed."""
    preds, trues, holidays = [], [], []
    model.eval()
    with torch.no_grad():
        for key, (sales, holiday, dates) in val_arrays.items():
            hist_raw = sales[:train_weeks][-lookback:]
            fut_holiday = holiday[train_weeks: train_weeks + horizon]
            fut_dates = dates[train_weeks: train_weeks + horizon]
            target_raw = sales[train_weeks: train_weeks + horizon]
            pred_raw = recursive_forecast(model, hist_raw, fut_holiday, fut_dates, lookback, horizon, n_blocks=1, device=device)
            pred_raw = np.clip(pred_raw, 0, None)
            preds.extend(pred_raw); trues.extend(target_raw); holidays.extend(fut_holiday)
    return wmae(trues, preds, holidays)


def train_model(train_ds, val_arrays, train_weeks, lookback, horizon, patch_len, stride, d_model, n_heads, n_layers,
                 learning_rate, max_epochs, patience, batch_size, device, verbose=False):
    """Trains with early stopping on val WMAE. Returns (model, best_val_wmae,
    best_epoch, history) — best_epoch reused later (Step 7/8) as a fixed
    training length once there's no more validation data to early-stop
    against, same convention as DLinear/TFT."""
    dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    model = PatchTST(lookback, horizon, patch_len=patch_len, stride=stride, d_model=d_model,
                      n_heads=n_heads, n_layers=n_layers, d_ff=d_model * 4, n_aux_channels=horizon * 3).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val, best_state, best_epoch, no_improve, history = np.inf, None, 0, 0, []
    for epoch in range(max_epochs):
        model.train()
        for hist_b, aux_b, target_b in dl:
            hist_b, aux_b, target_b = hist_b.to(device), aux_b.to(device), target_b.to(device)
            opt.zero_grad()
            is_holiday_b = aux_b[:, :horizon]
            loss = weighted_mae_loss(model(hist_b, aux_b), target_b, is_holiday_b)
            loss.backward()
            opt.step()
        val_wmae = evaluate_val_wmae(model, train_weeks, val_arrays, lookback, horizon, device)
        history.append(val_wmae)
        if val_wmae < best_val - 1e-3:
            best_val, best_epoch, no_improve = val_wmae, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                break
        if verbose and epoch % 10 == 0:
            print(f'  epoch {epoch:3d}: val WMAE = {val_wmae:.2f} (best {best_val:.2f} @ epoch {best_epoch})')
    model.load_state_dict(best_state)
    return model, best_val, best_epoch, history


def train_and_score_across_splits(cv_splits, patch_len, stride, d_model, n_heads, n_layers, learning_rate,
                                   max_epochs, patience, batch_size, device):
    """Trains one fresh model per split (mirrors every other notebook's
    per-fold/per-split pattern), returns (mean_val_wmae, per_split_wmaes,
    per_split_epochs)."""
    split_wmaes, split_epochs = [], []
    for s in cv_splits:
        _, val_wmae, epoch, _ = train_model(
            s['train_ds'], s['val_arrays'], s['train_end'], LOOKBACK, HORIZON_CV,
            patch_len, stride, d_model, n_heads, n_layers, learning_rate, max_epochs, patience, batch_size, device,
        )
        split_wmaes.append(val_wmae)
        split_epochs.append(epoch)
    return float(np.mean(split_wmaes)), split_wmaes, split_epochs


PATCH_LEN = 13
STRIDE = 13
N_HEADS = 4
N_LAYERS = 2
BASELINE_PARAMS = dict(d_model=16)
BASELINE_LR = 1e-3
MAX_EPOCHS = 200
PATIENCE = 15
BATCH_SIZE = 512

print('Training baseline PatchTST across all 3 splits...')
baseline_val_wmae, baseline_split_wmaes, baseline_split_epochs = train_and_score_across_splits(
    cv_splits, patch_len=PATCH_LEN, stride=STRIDE, n_heads=N_HEADS, n_layers=N_LAYERS,
    **BASELINE_PARAMS, learning_rate=BASELINE_LR, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, device=device,
)
print(f'Baseline mean val WMAE: {baseline_val_wmae:.2f}  (per-split: {[round(w, 1) for w in baseline_split_wmaes]}, '
      f'best epochs: {baseline_split_epochs})')

<a id='4'></a>
## 4. Model Architecture

`utils.patchtst.PatchTST`:

1. **RevIN**: each window's own lookback mean/std (`hist.mean(dim=1)`,
   `hist.std(dim=1).clamp_min(1e-5)`) normalizes that window; the model's
   output is denormalized with the same two numbers at the end. The
   `clamp_min` matters for real data: a series that's mostly gap-filled
   zeros within one window has std~0, and dividing by that unclamped would
   produce inf/NaN — verified directly against a constant (all-same-value)
   window during validation.
2. **Patching**: the normalized lookback (52 weeks) is split into
   `(52 - patch_len) / stride + 1 = 4` non-overlapping patches of 13 weeks
   each (`patch_len=13, stride=13`) via `Tensor.unfold` — verified to
   produce an exact, non-overlapping partition of the 52-week window.
3. **Patch embedding + positional encoding**: each 13-week patch is
   linearly projected to `d_model` dimensions, plus a learned positional
   embedding per patch position (so the model can tell "this is the most
   recent quarter" from "this is a year ago").
4. **Transformer encoder** (`nn.TransformerEncoder`, `n_layers` layers,
   `n_heads` attention heads) over the 4-patch sequence — attention over 4
   tokens instead of 52 raw timesteps is what keeps this cheap relative to
   a plain per-timestep transformer, and is the core idea the "worth 64
   words" title refers to.
5. **Flatten + aux + linear head**: the encoder's output across all patches
   is flattened into one vector, concatenated with the future `IsHoliday` +
   week-sin + week-cos covariates for the block being predicted (same
   `build_aux_features` layout DLinear uses), then one linear layer maps
   directly to the `horizon`-length forecast.

`patch_len`/`stride`/`n_heads`/`n_layers` are fixed at sensible defaults
(non-overlapping patches for simplicity, 4 heads and 2 layers as
commonly-used starting points for a dataset this size) rather than
grid-searched — `d_model` and `learning_rate` are the two hyperparameters
tuned in Step 5, since they're the ones most likely to trade off capacity
against overfitting risk for this amount of data.

<a id='5'></a>
## 5. Hyperparameter Tuning (manual grid, WMAE-scored)

Same grid shape as DLinear's — `d_model` and `learning_rate`, 9 combos:

| Param | Values |
|---|---|
| `d_model` | 16, 32, 64 |
| `learning_rate` | 1e-2, 3e-3, 1e-3 |

Each combo trains 3 independent models (one per split, early stopping
per-split, `patience=15`, `max_epochs=200`), ranked by mean WMAE across
splits — 27 fits total, same scale as DLinear's grid search. The winning
combo's 3 per-split `best_epoch` values get **median**-aggregated for
Step 7/8's fixed training length, same robust-epoch convention as
DLinear/TFT.

In [ ]:
param_grid = {
    'd_model': [16, 32, 64],
    'learning_rate': [1e-2, 3e-3, 1e-3],
}
param_names = list(param_grid.keys())
param_combos = list(itertools.product(*param_grid.values()))
print(f'{len(param_combos)} hyperparameter combinations x {len(cv_splits)} splits = {len(param_combos) * len(cv_splits)} fits')

In [ ]:
grid_results = []
for combo_idx, combo_values in enumerate(param_combos, start=1):
    params = dict(zip(param_names, combo_values))
    mean_wmae, split_wmaes, split_epochs = train_and_score_across_splits(
        cv_splits, patch_len=PATCH_LEN, stride=STRIDE, n_heads=N_HEADS, n_layers=N_LAYERS,
        **params, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, device=device,
    )
    grid_results.append({**params, 'mean_val_wmae': mean_wmae, 'split_wmaes': split_wmaes, 'split_epochs': split_epochs})
    print(f'[{combo_idx:2d}/{len(param_combos)}] {params} -> mean_val_wmae={mean_wmae:.2f}  '
          f'(per-split: {[round(w, 1) for w in split_wmaes]}, best epochs: {split_epochs})')

grid_results_df = pd.DataFrame(grid_results).sort_values('mean_val_wmae').reset_index(drop=True)
grid_results_df[['d_model', 'learning_rate', 'mean_val_wmae', 'split_wmaes', 'split_epochs']]

In [ ]:
best_params = {k: grid_results_df.loc[0, k] for k in param_names}
best_val_wmae = grid_results_df.loc[0, 'mean_val_wmae']
best_split_epochs = grid_results_df.loc[0, 'split_epochs']
median_best_epoch = int(round(np.median(best_split_epochs)))

print('Best params:', best_params)
print(f'Best mean val WMAE: {best_val_wmae:.2f} (vs baseline: {baseline_val_wmae:.2f})')
print(f'Best epochs per split: {best_split_epochs} -> median {median_best_epoch} '
      f'(reused as a fixed training length once Step 7/8 have no validation set left)')

<a id='6'></a>
## 6. MLflow Logging (DagsHub-hosted)

Same DagsHub-hosted MLflow server as every other notebook, separate
experiment `PatchTST_Training`. Five runs, same structure as DLinear's:

1. `PatchTST_Cleaning` — data shape and the local train/test split (Step 2)
2. `PatchTST_Windowing` — sequence-construction config and per-split dataset sizes (Step 3)
3. `PatchTST_CV_Tuning` — the 27-fit grid search (Step 5), batched summary
4. `PatchTST_CV` — genuinely incremental: retrains the winning config once more, logging `val_wmae` after every epoch as it happens
5. `PatchTST_Final_Fit` — the full pipeline (Step 8)

Plot artifacts are skipped (params/metrics only), same choice as the other
notebooks.

In [ ]:
import dagshub

dagshub.init(repo_owner='tgela23', repo_name='ml-final-project', mlflow=True)

import mlflow
mlflow.set_experiment('PatchTST_Training')
print('tracking uri:', mlflow.get_tracking_uri())

**Run 1 — `PatchTST_Cleaning`**

In [ ]:
with mlflow.start_run(run_name='PatchTST_Cleaning'):
    mlflow.log_param('train_csv_shape', str(train.shape))
    mlflow.log_param('test_csv_shape', str(test.shape))
    mlflow.log_param('train_date_range', f'{train.Date.min().date()} -> {train.Date.max().date()}')

    mlflow.log_param('local_test_holdout_weeks', 52)
    mlflow.log_param('local_train_date_range', f'{local_train_raw.Date.min().date()} -> {local_train_raw.Date.max().date()}')
    mlflow.log_param('local_test_date_range', f'{local_test_raw.Date.min().date()} -> {local_test_raw.Date.max().date()}')

    mlflow.log_metric('n_train_rows', len(train))
    mlflow.log_metric('n_local_train_rows', len(local_train_raw))
    mlflow.log_metric('n_local_test_rows', len(local_test_raw))
    mlflow.log_metric('n_stores', train['Store'].nunique())
    mlflow.log_metric('n_depts', train['Dept'].nunique())
    mlflow.log_metric('n_series', train[['Store', 'Dept']].drop_duplicates().shape[0])

print('PatchTST_Cleaning run logged.')

**Run 2 — `PatchTST_Windowing`**

In [ ]:
with mlflow.start_run(run_name='PatchTST_Windowing'):
    mlflow.log_param('lookback', LOOKBACK)
    mlflow.log_param('horizon_cv', HORIZON_CV)
    mlflow.log_param('horizon_final', HORIZON_FINAL)
    mlflow.log_param('patch_len', PATCH_LEN)
    mlflow.log_param('stride', STRIDE)
    mlflow.log_param('n_heads', N_HEADS)
    mlflow.log_param('n_layers', N_LAYERS)
    mlflow.log_param('cv_split_train_weeks', str(CV_SPLIT_TRAIN_WEEKS))

    for i, s in enumerate(cv_splits):
        mlflow.log_metric(f'split{i}_train_end', s['train_end'])
        mlflow.log_metric(f'split{i}_n_windows', len(s['train_ds']))
    mlflow.log_metric('baseline_val_wmae', baseline_val_wmae)

print('PatchTST_Windowing run logged.')

**Run 3 — `PatchTST_CV_Tuning`**: batched summary of the 27-fit grid search.

In [ ]:
with mlflow.start_run(run_name='PatchTST_CV_Tuning'):
    mlflow.log_param('param_grid', str(param_grid))

    for i, row in grid_results_df.iterrows():
        mlflow.log_metric('combo_mean_val_wmae', row['mean_val_wmae'], step=i)

    mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
    mlflow.log_metric('best_mean_val_wmae', best_val_wmae)
    mlflow.log_metric('median_best_epoch', median_best_epoch)

print('PatchTST_CV_Tuning run logged.')

**Run 4 — `PatchTST_CV`**: genuinely incremental companion to
`PatchTST_CV_Tuning` — retrains the winning hyperparameters across all 3
splits from scratch and logs each split's `val_wmae` after every epoch
inside the open run, as it happens.

In [ ]:
with mlflow.start_run(run_name='PatchTST_CV'):
    mlflow.log_params(best_params)
    mlflow.log_param('max_epochs', MAX_EPOCHS)
    mlflow.log_param('patience', PATIENCE)
    mlflow.log_param('batch_size', BATCH_SIZE)

    tuned_split_wmaes, tuned_split_epochs = [], []
    for i, s in enumerate(cv_splits):
        model_i, val_wmae, epoch, history = train_model(
            s['train_ds'], s['val_arrays'], s['train_end'], LOOKBACK, HORIZON_CV,
            PATCH_LEN, STRIDE, best_params['d_model'], N_HEADS, N_LAYERS, best_params['learning_rate'],
            MAX_EPOCHS, PATIENCE, BATCH_SIZE, device,
        )
        for e, wmae_e in enumerate(history):
            mlflow.log_metric(f'split{i}_val_wmae', wmae_e, step=e)
        tuned_split_wmaes.append(val_wmae)
        tuned_split_epochs.append(epoch)
    tuned_val_wmae = float(np.mean(tuned_split_wmaes))

    mlflow.log_metric('final_mean_val_wmae', tuned_val_wmae)
    for i, e in enumerate(tuned_split_epochs):
        mlflow.log_metric(f'final_split{i}_best_epoch', e)

print(f'PatchTST_CV run logged. mean val WMAE = {tuned_val_wmae:.2f} '
      f'(per-split: {[round(w, 1) for w in tuned_split_wmaes]}, best epochs: {tuned_split_epochs})')

<a id='7'></a>
## 7. Plots

**A real constraint PatchTST hits that TFT doesn't**: TFT's
`TimeSeriesDataSet` supports a variable-length encoder (`min_encoder_length`
to `max_encoder_length`), so it could still find valid training windows on
`local_train_raw` alone even with a 52-week direct decoder. `PatchTST`'s
lookback is architecturally fixed (patches are cut from a single fixed-size
window) — `LOOKBACK=52` and `HORIZON_FINAL=52` together need 104 weeks,
more than `local_train_raw`'s 91. So Steps 7/8 use `LOOKBACK_FINAL=39`
instead (`91 - 52 = 39`, the largest lookback that still leaves room for a
direct 52-week decoder inside 91 weeks — and, conveniently, still divides
evenly into 3 non-overlapping 13-week patches). This means the holdout-eval
model has *less* context per prediction (39 weeks vs. the 52 tuned on) and,
worse, exactly **one** training window per series for Step 7 specifically
(`91 - 39 - 52 + 1 = 1`) — thin, but the same order of magnitude as
DLinear's thinnest CV split, and validated to train without error. Step 8
(all of `train.csv`, 143 weeks) doesn't have this problem (`143 - 39 - 52 +
1 = 53` windows/series) — plenty of data there.

`N_EPOCHS_FINAL` is `median(tuned_split_epochs) + 1` from Run 4 — same
convention as DLinear/TFT. All plots saved to `plots/` with a `_patchtst`
suffix.

In [ ]:
LOOKBACK_FINAL = 39  # 91 (local_train_raw) - 52 (HORIZON_FINAL) = 39, and 39 / 13 = 3 patches exactly
N_EPOCHS_FINAL = median_best_epoch + 1

eval_panel = build_full_calendar_panel(local_train_raw)
eval_arrays = series_arrays_from_panel(eval_panel)
eval_train_ds = PatchTSTWindowDataset(eval_arrays, LOOKBACK_FINAL, HORIZON_FINAL)

print(f'Training final holdout-eval model for {N_EPOCHS_FINAL} fixed epochs '
      f'(median of {tuned_split_epochs} + 1) on {len(eval_train_ds)} windows '
      f'(1 per series, {LOOKBACK_FINAL}-week lookback)...')

eval_dl = DataLoader(eval_train_ds, batch_size=BATCH_SIZE, shuffle=True)
holdout_model = PatchTST(LOOKBACK_FINAL, HORIZON_FINAL, patch_len=PATCH_LEN, stride=STRIDE,
                          d_model=best_params['d_model'], n_heads=N_HEADS, n_layers=N_LAYERS,
                          d_ff=best_params['d_model'] * 4, n_aux_channels=HORIZON_FINAL * 3).to(device)
opt = torch.optim.Adam(holdout_model.parameters(), lr=best_params['learning_rate'])
holdout_model.train()
for _epoch in range(N_EPOCHS_FINAL):
    for hist_b, aux_b, target_b in eval_dl:
        hist_b, aux_b, target_b = hist_b.to(device), aux_b.to(device), target_b.to(device)
        opt.zero_grad()
        is_holiday_b = aux_b[:, :HORIZON_FINAL]
        loss = weighted_mae_loss(holdout_model(hist_b, aux_b), target_b, is_holiday_b)
        loss.backward()
        opt.step()

holdout_panel = build_full_calendar_panel(pd.concat([local_train_raw, local_test_raw], ignore_index=True))
holdout_arrays = series_arrays_from_panel(holdout_panel)

In [ ]:
pred_rows = []
holdout_model.eval()
for key, (sales, holiday, dates_arr) in holdout_arrays.items():
    hist_raw = sales[:len(local_train_dates)][-LOOKBACK_FINAL:]
    fut_holiday = holiday[len(local_train_dates): len(local_train_dates) + HORIZON_FINAL]
    fut_dates = dates_arr[len(local_train_dates): len(local_train_dates) + HORIZON_FINAL]
    pred_raw = recursive_forecast(holdout_model, hist_raw, fut_holiday, fut_dates, LOOKBACK_FINAL, HORIZON_FINAL, n_blocks=1, device=device)
    pred_raw = np.clip(pred_raw, 0, None)
    test_dates = dates_arr[len(local_train_dates): len(local_train_dates) + HORIZON_FINAL]
    test_actual = sales[len(local_train_dates): len(local_train_dates) + HORIZON_FINAL]
    test_holiday = holiday[len(local_train_dates): len(local_train_dates) + HORIZON_FINAL].astype(bool)
    for d, a, p, h in zip(test_dates, test_actual, pred_raw, test_holiday):
        pred_rows.append((key[0], key[1], pd.Timestamp(d), a, p, h))

pred_df = pd.DataFrame(pred_rows, columns=['Store', 'Dept', 'Date', 'Actual', 'Predicted', 'IsHoliday'])
pred_df['Residual'] = pred_df['Actual'] - pred_df['Predicted']

holdout_wmae = wmae(pred_df['Actual'], pred_df['Predicted'], pred_df['IsHoliday'])
print(f'Local-test holdout WMAE (tuned model, direct 52-week forecast, no recursion): {holdout_wmae:.2f}')

**Plot 1 — Actual vs. predicted over time**, the same 3 sample
Store/Dept series used in every other notebook, across the local-test
holdout.

In [ ]:
sample_combos = [(1, 1), (1, 72), (20, 1)]

fig, axes = plt.subplots(len(sample_combos), 1, figsize=(11, 9), sharex=True)
for ax, (store, dept) in zip(axes, sample_combos):
    sub = pred_df[(pred_df['Store'] == store) & (pred_df['Dept'] == dept)].sort_values('Date')
    ax.plot(sub['Date'], sub['Actual'], label='Actual', marker='o', markersize=3)
    ax.plot(sub['Date'], sub['Predicted'], label='Predicted', marker='x', markersize=3)
    holiday_dates = sub.loc[sub['IsHoliday'], 'Date']
    for hd in holiday_dates:
        ax.axvline(hd, color='gray', alpha=0.3, linestyle='--')
    ax.set_title(f'Store {store}, Dept {dept}')
    ax.legend()
ax.set_xlabel('Date')
fig.suptitle('PatchTST: actual vs. predicted — local-test holdout (dashed lines = holiday weeks)')
plt.tight_layout()
plt.savefig('plots/actual_vs_predicted_timeseries_patchtst.png', dpi=150, bbox_inches='tight')
plt.show()

**Plot 2 — Patch & covariate importance**, PatchTST's analogue of a
feature importance plot: the output head's weight magnitude, averaged
across the horizon and grouped by which input block it reads from — each
of the (here, 3) patches, plus the future `IsHoliday`/week-sin/week-cos
covariate groups.

In [ ]:
def patch_and_aux_importance(model, horizon):
    W = model.head.weight.detach().cpu().abs()  # (horizon, num_patches*d_model + horizon*3)
    num_patches = model.num_patches
    d_model = model.patch_embed.out_features
    patch_importance = [W[:, p * d_model:(p + 1) * d_model].mean().item() for p in range(num_patches)]
    aux = W[:, num_patches * d_model:]
    aux_importance = [aux[:, :horizon].mean().item(), aux[:, horizon:2 * horizon].mean().item(), aux[:, 2 * horizon:].mean().item()]
    labels = [f'Patch {p + 1}\n(weeks {p * PATCH_LEN + 1}-{(p + 1) * PATCH_LEN})' for p in range(num_patches)]
    labels += ['IsHoliday', 'week-sin', 'week-cos']
    return labels, patch_importance + aux_importance

labels, importances = patch_and_aux_importance(holdout_model, HORIZON_FINAL)
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4C72B0'] * holdout_model.num_patches + ['#DD8452', '#DD8452', '#DD8452']
ax.bar(labels, importances, color=colors)
ax.set_ylabel('Mean |head weight| across the horizon')
ax.set_title('PatchTST: patch & future-covariate importance (blue = history patches, orange = calendar covariates)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('plots/feature_importance_tuned_patchtst.png', dpi=150)
plt.show()

**Plot 3 — WMAE by stage**: baseline (untuned, mean across splits) vs.
tuned (mean across splits) vs. final holdout.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
stages = ['Baseline\n(mean, 3 splits)', 'Tuned\n(mean, 3 splits)', 'Final\n(holdout)']
values = [baseline_val_wmae, tuned_val_wmae, holdout_wmae]
colors = ['#B0B0B0', '#4C72B0', '#55A868']
ax.bar(stages, values, color=colors)
for i, v in enumerate(values):
    ax.text(i, v + max(values) * 0.01, f'{v:.0f}', ha='center')
ax.set_ylabel('WMAE')
ax.set_title('WMAE by stage — baseline vs. tuned vs. final holdout (PatchTST)')
plt.tight_layout()
plt.savefig('plots/wmae_by_fold_patchtst.png', dpi=150)
plt.show()

**Plot 4 — Residual distribution** on the local-test holdout.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pred_df['Residual'], bins=60, color='#4C72B0', alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Residual (Actual - Predicted)')
ax.set_title('Residual distribution — local-test holdout (PatchTST)')
plt.tight_layout()
plt.savefig('plots/residual_distribution_patchtst.png', dpi=150)
plt.show()

**Plot 5 — Actual vs. predicted, holiday vs. non-holiday weeks**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for is_holiday, color, label in [(False, '#4C72B0', 'Non-holiday'), (True, '#C44E52', 'Holiday')]:
    sub = pred_df[pred_df['IsHoliday'] == is_holiday]
    ax.scatter(sub['Actual'], sub['Predicted'], alpha=0.3, s=10, color=color, label=label)
lims = [0, pred_df[['Actual', 'Predicted']].to_numpy().max() * 1.05]
ax.plot(lims, lims, color='black', linewidth=1, linestyle='--')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title('Actual vs. Predicted — Holiday vs. Non-holiday weeks (PatchTST)')
ax.legend()
plt.tight_layout()
plt.savefig('plots/actual_vs_predicted_holiday_patchtst.png', dpi=150)
plt.show()

<a id='8'></a>
## 8. Full Pipeline

`utils.patchtst.PatchTSTForecastPipeline` is the PatchTST analogue of
`DLinearForecastPipeline`: `fit()` stores every series' full history;
`predict()` takes bare `Store/Dept/Date/IsHoliday` rows (e.g. `test.csv`
exactly as downloaded) and returns `Weekly_Sales` predictions — no manual
feature work by the caller, and no per-series stats to maintain (RevIN
handles normalization per-window, inside the model).

Fit on **all of `train.csv`** (143 weeks, all 3,331 series — everything
before the real Kaggle `test.csv`, now that tuning/plots are done), same
`LOOKBACK_FINAL=39`/`HORIZON_FINAL=52` as Step 7's holdout model (143
weeks comfortably fits `39 + 52 = 91` with room to spare — `53` windows per
series, unlike Step 7's constrained `1`), for the same `N_EPOCHS_FINAL`
fixed epochs.

In [ ]:
full_panel = build_full_calendar_panel(train)
full_arrays = series_arrays_from_panel(full_panel)
full_train_ds = PatchTSTWindowDataset(full_arrays, LOOKBACK_FINAL, HORIZON_FINAL)
n_series_full = len(full_panel[['Store', 'Dept']].drop_duplicates())

print(f'{len(full_train_ds)} training windows across all of train.csv '
      f'({len(full_train_ds) / n_series_full:.0f} per series, {n_series_full} series)')

print(f'Training final model for {N_EPOCHS_FINAL} fixed epochs...')
full_dl = DataLoader(full_train_ds, batch_size=BATCH_SIZE, shuffle=True)
final_model = PatchTST(LOOKBACK_FINAL, HORIZON_FINAL, patch_len=PATCH_LEN, stride=STRIDE,
                        d_model=best_params['d_model'], n_heads=N_HEADS, n_layers=N_LAYERS,
                        d_ff=best_params['d_model'] * 4, n_aux_channels=HORIZON_FINAL * 3).to(device)
opt = torch.optim.Adam(final_model.parameters(), lr=best_params['learning_rate'])
final_model.train()
for _epoch in range(N_EPOCHS_FINAL):
    for hist_b, aux_b, target_b in full_dl:
        hist_b, aux_b, target_b = hist_b.to(device), aux_b.to(device), target_b.to(device)
        opt.zero_grad()
        is_holiday_b = aux_b[:, :HORIZON_FINAL]
        loss = weighted_mae_loss(final_model(hist_b, aux_b), target_b, is_holiday_b)
        loss.backward()
        opt.step()

full_pipeline = PatchTSTForecastPipeline(final_model, LOOKBACK_FINAL, HORIZON_FINAL, device=device)
full_pipeline.fit(train)
print('PatchTSTForecastPipeline fit on all of train.csv.')

**Confirm it truly takes raw input**: predict on bare rows straight from
`test.csv` — unmerged, no history/lag/rolling computed by the caller.

In [ ]:
raw_sample = test.head(5)
print('Raw input columns (exactly test.csv, nothing pre-computed):', raw_sample.columns.tolist())

raw_preds = full_pipeline.predict(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']])
print()
print('Predictions:', raw_preds)

**Save to MLflow (DagsHub model registry)** inside `PatchTST_Final_Fit`.
`mlflow.pyfunc` (not `mlflow.pytorch`), same reasoning as every other
notebook: the saved artifact needs to be the whole raw-input-to-prediction
path, not just the bare `nn.Module`.

In [ ]:
from mlflow.models import infer_signature


class PatchTSTPyfuncWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, pipeline):
        self.pipeline = pipeline

    def predict(self, context, model_input, params=None):
        return self.pipeline.predict(model_input)


signature = infer_signature(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']], raw_preds)

with mlflow.start_run(run_name='PatchTST_Final_Fit'):
    mlflow.log_params(best_params)
    mlflow.log_param('lookback_final', LOOKBACK_FINAL)
    mlflow.log_param('horizon_final', HORIZON_FINAL)
    mlflow.log_param('n_epochs_final', N_EPOCHS_FINAL)
    mlflow.log_metric('local_test_holdout_wmae', holdout_wmae)
    mlflow.log_metric('n_series_trained', n_series_full)

    mlflow.pyfunc.log_model(
        artifact_path='model',
        python_model=PatchTSTPyfuncWrapper(full_pipeline),
        signature=signature,
        input_example=raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']],
    )

print('PatchTST_Final_Fit run logged, pipeline saved to MLflow model registry.')

**Also save locally** under `models/`, via `torch.save` (a plain
`nn.Module` state_dict, unlike TFT's Lightning checkpoint) plus the raw
history `PatchTSTForecastPipeline` needs.

In [ ]:
import os

os.makedirs('models', exist_ok=True)
torch.save({
    'model_state_dict': full_pipeline.model.state_dict(),
    'lookback': LOOKBACK_FINAL,
    'horizon': HORIZON_FINAL,
    'patch_len': PATCH_LEN,
    'stride': STRIDE,
    'd_model': best_params['d_model'],
    'n_heads': N_HEADS,
    'n_layers': N_LAYERS,
    'history_': full_pipeline.history_,
    'last_date_': full_pipeline.last_date_,
}, 'models/patchtst_pipeline.pt')
print('Saved to models/patchtst_pipeline.pt')

In [ ]:
checkpoint = torch.load('models/patchtst_pipeline.pt', weights_only=False)

reloaded_model = PatchTST(
    lookback=checkpoint['lookback'], horizon=checkpoint['horizon'],
    patch_len=checkpoint['patch_len'], stride=checkpoint['stride'], d_model=checkpoint['d_model'],
    n_heads=checkpoint['n_heads'], n_layers=checkpoint['n_layers'], d_ff=checkpoint['d_model'] * 4,
    n_aux_channels=checkpoint['horizon'] * 3,
).to(device)
reloaded_model.load_state_dict(checkpoint['model_state_dict'])

reloaded_pipeline = PatchTSTForecastPipeline(reloaded_model, checkpoint['lookback'], checkpoint['horizon'], device=device)
reloaded_pipeline.history_ = checkpoint['history_']
reloaded_pipeline.last_date_ = checkpoint['last_date_']

reloaded_preds = reloaded_pipeline.predict(raw_sample[['Store', 'Dept', 'Date', 'IsHoliday']])
print('Reloaded-pipeline predictions match:', np.allclose(reloaded_preds, raw_preds))